# Diff-MoE on TinyStories -- Kaggle T4 x2 training notebook

Runs one config from `docs/plan.md`'s 2x2 ablation (standard/differential attention x dense/MoE FFN) on Kaggle's dual-T4 notebook, using both GPUs via DDP.

**Before running**: set Notebook Settings -> Accelerator = **GPU T4 x2**.

**Session limits**: 12h max, ~30 GPU-h/week (same quota whether you pick T4 x1 or x2). Checkpoints save every `ckpt_freq` steps to `/kaggle/working/checkpoints` -- copy that folder to a Kaggle Dataset before your session ends so the next session can resume.

**Workflow per run**: (1) clone repo, (2) prepare tokenized data once and re-use across sessions via a Kaggle Dataset, (3) throughput probe on both GPUs, (4) train with DDP, (5) inspect metrics + report.

## 1. Setup

In [9]:
!git clone -b rebuild https://github.com/ramprasathk07/Differential-MOE.git /kaggle/working/repo
%cd /kaggle/working/repo

# Checkout the specific commit
# !git checkout fb66a47b4dad44e8b0f7d6144f0cfebb5681c0a5

# # (Optional) Verify you're on the correct commit
# !git rev-parse HEAD

!pip install -q -r requirements.txt

fatal: destination path '/kaggle/working/repo' already exists and is not an empty directory.
/kaggle/working/repo


In [10]:
import torch
n_gpu = torch.cuda.device_count()
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', n_gpu)
for i in range(n_gpu):
    print(f'  cuda:{i}', torch.cuda.get_device_name(i))
if n_gpu < 2:
    print('\nWARNING: fewer than 2 GPUs visible -- set Notebook Settings > Accelerator = GPU T4 x2, '
          'then Session > Restart & Run All. The --ddp cells below need nproc_per_node=2 to match n_gpu.')

CUDA available: True
GPU count: 2
  cuda:0 Tesla T4
  cuda:1 Tesla T4


## 2. Run settings

Change these per run -- no code edits needed elsewhere in the notebook.

In [11]:
CONFIG = 'configs/a_dense.yaml'  # a_dense / a_diff / a_moe / a_diffmoe / b_final
WANDB_PROJECT = 'a_diffmoe'  # change freely per experiment batch, no code changes needed
USE_WANDB = True
N_GPU = 2  # matches Accelerator = GPU T4 x2; set to 1 if you picked the single-T4 option

DATA_DIR = '/kaggle/working/data'  # switch to '/kaggle/input/<dataset-name>' once tokenized once and re-uploaded

run_name = CONFIG.split('/')[-1].replace('.yaml', '')
wandb_flag = f'--wandb --wandb_project {WANDB_PROJECT}' if USE_WANDB else ''
ddp_prefix = f'torchrun --standalone --nproc_per_node={N_GPU}' if N_GPU > 1 else 'python'
ddp_flag = '--ddp' if N_GPU > 1 else ''
print('run_name:', run_name)
print('launch prefix:', ddp_prefix)

run_name: a_dense
launch prefix: torchrun --standalone --nproc_per_node=2


## 3. Tokenizer + data

Run once, then attach the output as a Kaggle Dataset ("New Dataset" from notebook output) so later sessions can skip straight to training via `DATA_DIR` pointing at `/kaggle/input/<dataset-name>`. Tokenizing is CPU-only -- no benefit from N_GPU here.

In [12]:
import os
if not os.path.exists(f'{DATA_DIR}/train.bin'):
    # adaptive vocab sweep (docs/plan.md SS2) -- run once, read the fertility table, pick a vocab_size
    !python -m src.data.train_tokenizer --sweep --candidates 2048 4096 8192 16384 --max_stories 50000
else:
    print('data already prepared at', DATA_DIR)

usage: train_tokenizer.py [-h] [--sweep]
                          [--candidates CANDIDATES [CANDIDATES ...]]
                          [--vocab_size VOCAB_SIZE]
                          [--track {strict-small,strict}] [--out OUT]
                          [--max_lines_per_domain MAX_LINES_PER_DOMAIN]
                          [--eval_max_lines_per_domain EVAL_MAX_LINES_PER_DOMAIN]
train_tokenizer.py: error: unrecognized arguments: --max_stories 50000


In [13]:
VOCAB_SIZE = 4096  # set from the sweep table above -- must match the vocab_size in CONFIG's yaml

if not os.path.exists(f'{DATA_DIR}/train.bin'):
    !python -m src.data.train_tokenizer --vocab_size {VOCAB_SIZE} --out {DATA_DIR}/tokenizer.json --max_stories 0
    !python -m src.data.prepare --tokenizer {DATA_DIR}/tokenizer.json --out_dir {DATA_DIR} --max_stories 0

usage: train_tokenizer.py [-h] [--sweep]
                          [--candidates CANDIDATES [CANDIDATES ...]]
                          [--vocab_size VOCAB_SIZE]
                          [--track {strict-small,strict}] [--out OUT]
                          [--max_lines_per_domain MAX_LINES_PER_DOMAIN]
                          [--eval_max_lines_per_domain EVAL_MAX_LINES_PER_DOMAIN]
train_tokenizer.py: error: unrecognized arguments: --max_stories 0
usage: prepare.py [-h] [--tokenizer TOKENIZER] [--out_dir OUT_DIR]
                  [--track {strict-small,strict}]
                  [--max_lines_per_domain MAX_LINES_PER_DOMAIN]
prepare.py: error: unrecognized arguments: --max_stories 0


In [14]:
# import wandb

# wandb_key = "wandb_v1_B1ZUHptC9pjBCEX0BCs12V7xWi9_6NcX7Cq  GwecxYgdFZeIZh1XBwj9cEZTIGGnnrrcbdcS3tO1SW"
# # Login
# wandb.login(key=wandb_key)

In [15]:
from kaggle_secrets import UserSecretsClient
import wandb

# Get API key from Kaggle Secrets
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb")  # Replace with your secret name if different

# Login
wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

## 4. Throughput probe (docs/plan.md Phase 3)

Runs ~100 steps across both GPUs and reports tok/s so the token budget / wall-clock estimate is measured, not guessed. `batch_size` in the config is **per-GPU** -- with `N_GPU=2` the effective global batch doubles automatically (see `report.json`'s `effective_global_batch_tokens`).

In [16]:
!{ddp_prefix} -m src.train --config {CONFIG} --data_dir {DATA_DIR} --out_dir /kaggle/working/checkpoints {ddp_flag} --max_steps 100

W0720 07:14:20.481000 228 torch/distributed/run.py:852] 
W0720 07:14:20.481000 228 torch/distributed/run.py:852] *****************************************
W0720 07:14:20.481000 228 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0720 07:14:20.481000 228 torch/distributed/run.py:852] *****************************************
[W720 07:14:20.243362383 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W720 07:14:22.265836360 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W720 07:14:22.279879933 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can sp

## 5. Full training run

Re-running this cell auto-resumes from `checkpoints/<run_name>/last.pt` if it exists -- safe to re-run after a Kaggle session restart. Rank-0-only logging/checkpointing/wandb is handled inside `src/train.py`; nothing extra needed here.

In [17]:
!{ddp_prefix} -m src.train --config {CONFIG} --data_dir {DATA_DIR} --out_dir /kaggle/working/checkpoints {ddp_flag} {wandb_flag}

W0720 07:14:28.516000 261 torch/distributed/run.py:852] 
W0720 07:14:28.516000 261 torch/distributed/run.py:852] *****************************************
W0720 07:14:28.516000 261 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0720 07:14:28.516000 261 torch/distributed/run.py:852] *****************************************
[W720 07:14:28.262513225 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W720 07:14:30.302896289 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W720 07:14:30.327537833 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can sp

## 6. Inspect metrics + report

In [18]:
import json
with open(f'/kaggle/working/checkpoints/{run_name}/report.json') as f:
    print(json.dumps(json.load(f), indent=2))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/checkpoints/a_dense/report.json'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f'/kaggle/working/checkpoints/{run_name}/metrics.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.dropna(subset=['train_loss']).plot(x='step', y='train_loss', ax=axes[0], title='train loss')
df.dropna(subset=['val_nll']).plot(x='step', y='val_nll', ax=axes[1], title='val NLL', marker='o')
plt.tight_layout()
plt.show()
df.tail(10)

## 7. Params table (for README / blog tables -- never hand-compute)

In [ ]:
!python -m src.params --config configs/a_dense.yaml configs/a_diff.yaml configs/a_moe.yaml configs/a_diffmoe.yaml

## 8. Persist checkpoints across sessions

Kaggle wipes `/kaggle/working` between sessions unless saved as notebook output. Run this before your 12h session ends, then attach the resulting Dataset as input next session and point `DATA_DIR` / `--out_dir` at it to resume.

In [ ]:
# Notebook output already includes anything under /kaggle/working when you 'Save Version'.
# For a lighter artifact, keep only what's needed to resume + report:
!ls -la /kaggle/working/checkpoints/{run_name}/
print('best/ holds only the top-2 checkpoints (max_best_checkpoints in config); last.pt is always kept for resume.')